# Scribble notebook for visualizing layers, masks etc on the go

This notebook serves as a quick notedown for viewing masks and other outputs. Running it locally will probably lead to a Kernel crash (by experience when working with my local VS studio lab), therefore i try to run this in my Google colab & export the outputs into the separate folder "results" of this project repository.  

IMPORTANT: need to check whether colab can run this notebook's library loads or whether i need to add a pip install !!!!???

In [ ]:
from google.colab import files

files.upload()
# upload the building_vegetation_mask.tif and the combined_mask.tif files to the Colab environment.

In [ ]:
# install required packages
!pip install rasterio folium pillow

In [ ]:
# Read and downsample the raster
import rasterio
from rasterio.enums import Resampling

raster_path = "building_plus_vegetation.tif"

with rasterio.open(raster_path) as src:

    # Downsample by factor 10
    scale_factor = 10

    downsampled = src.read(
        1,
        out_shape=(
            src.height // scale_factor,
            src.width // scale_factor
        ),
        resampling=Resampling.nearest
    )

    bounds = src.bounds

In [ ]:
# Verify the size reduction
print("Original size:", src.height, src.width)
print("Downsampled size:", downsampled.shape)

In [ ]:
import numpy as np
from PIL import Image

rgb = np.zeros(
    (downsampled.shape[0], downsampled.shape[1], 3),
    dtype=np.uint8
)

rgb[downsampled == 0] = [255, 255, 255]  # background
rgb[downsampled == 1] = [255, 0, 0]      # buildings
rgb[downsampled == 2] = [0, 255, 0]      # building + vegetation

Image.fromarray(rgb).save("combined_mask_preview.png")

In [ ]:
# Display in a Leaflet map
import folium

center_lat = (bounds.top + bounds.bottom) / 2
center_lon = (bounds.left + bounds.right) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="CartoDB positron"
)

folium.raster_layers.ImageOverlay(
    image="combined_mask_preview.png",
    bounds=[
        [bounds.bottom, bounds.left],
        [bounds.top, bounds.right]
    ],
    opacity=0.8
).add_to(m)

folium.LayerControl().add_to(m)

m